# Gradia.AI: Exploratory Data Analysis of HAM10000

This notebook studies the dataset **before** any model is trained. Every finding here feeds a decision in the training notebook (how to split, which classes need help, which metadata can be used, how to preprocess).

**What it checks**

1. Metadata structure and data quality (missing values, duplicates, missing files)
2. Class balance and the "always predict nevus" baseline
3. Lesion-level structure, and how much the old `TRAIN.ipynb` split leaked
4. `dx_type`, and why it must never be a model input
5. Age, sex and body site per class (candidates for metadata fusion)
6. Image sizes, colour statistics, dark dermoscope corners, near-duplicate images
7. Sample images per class
8. What the old training vs inference preprocessing actually shows the model

**How to run:** Runtime → Run all. No GPU needed. The image pass reads all 10,015 files once (about 5–10 minutes from Google Drive) and caches the results, so re-runs are fast.

**Outputs** are saved to `MyDrive/eda_outputs/`: every figure as a PNG, the per-image statistics CSV, and `eda_summary.json` with the key numbers.

## 0. Setup

In [4]:
import os, json, shutil, random, warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

# ── Where the data lives ─────────────────────────────────────────────────────
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive"
else:
    DRIVE_DIR = os.environ.get("GRADIA_DATA_ROOT", ".")

IMAGES_DIR = f"{DRIVE_DIR}/HAM10000"                       # same paths as TRAIN.ipynb
CSV_PATH   = f"{DRIVE_DIR}/HAM10000/HAM10000_metadata.csv"
OUT_DIR    = f"{DRIVE_DIR}/eda_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Options ──────────────────────────────────────────────────────────────────
COPY_TO_LOCAL = IN_COLAB   # copy images to the Colab disk once: much faster reads than Drive
SEED = 42
random.seed(SEED); np.random.seed(SEED)

CLASS_NAMES = {
    "nv": "Melanocytic nevus", "mel": "Melanoma", "bkl": "Benign keratosis",
    "bcc": "Basal cell carcinoma", "akiec": "Actinic keratosis / Bowen's",
    "vasc": "Vascular lesion", "df": "Dermatofibroma",
}
MALIGNANT = {"mel", "bcc", "akiec"}

def boxplot(ax, data, labels):
    # matplotlib >= 3.9 renamed `labels` to `tick_labels`
    try:
        return ax.boxplot(data, tick_labels=labels)
    except TypeError:
        return ax.boxplot(data, labels=labels)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
FIGS = []
def save(name):
    path = f"{OUT_DIR}/{name}.png"
    plt.savefig(path, bbox_inches="tight", dpi=130)
    FIGS.append(path)
    plt.show()

SUMMARY = {}   # key numbers collected along the way, saved at the end
print("Images folder:", IMAGES_DIR)
print("Outputs folder:", OUT_DIR)

Images folder: ./HAM10000
Outputs folder: ./eda_outputs


In [ ]:
# Optional: copy images from Drive to the local Colab disk (one slow pass, then every read is fast)
if COPY_TO_LOCAL:
    LOCAL_DIR = "/content/ham_local"
    if not os.path.exists(LOCAL_DIR):
        print("Copying images to local disk (one time, a few minutes)...")
        shutil.copytree(IMAGES_DIR, LOCAL_DIR)
    IMAGES_DIR = LOCAL_DIR
print("Reading images from:", IMAGES_DIR)

## 1. Metadata structure

`HAM10000_metadata.csv` has one row per **image**. Columns:

| Column | Meaning |
|---|---|
| `lesion_id` | The physical lesion. **One lesion can have several images.** |
| `image_id` | Image file name without `.jpg` |
| `dx` | Diagnosis: the label we predict |
| `dx_type` | How the diagnosis was confirmed (histopathology, follow-up, expert consensus, confocal) |
| `age`, `sex`, `localization` | Patient and body-site metadata |

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes.to_string())
display(df.head())
SUMMARY["n_images"] = int(len(df))
SUMMARY["columns"] = list(df.columns)

## 2. Data quality

Missing values come in two forms here: real `NaN`s, and the string `unknown`. Both matter for metadata fusion, because the model needs a value for every input.

In [ ]:
quality = pd.DataFrame({
    "NaN": df.isna().sum(),
    "'unknown'": (df.astype(str).apply(lambda c: c.str.lower()) == "unknown").sum(),
})
quality["% missing (either)"] = ((quality["NaN"] + quality["'unknown'"]) / len(df) * 100).round(2)
display(quality)

print("Duplicate image_id rows:", int(df["image_id"].duplicated().sum()))
print("Fully duplicated rows:  ", int(df.duplicated().sum()))
print("Age range:", df["age"].min(), "to", df["age"].max(), "| rows with age == 0:", int((df["age"] == 0).sum()))

# Does any lesion carry two different diagnoses? (should be 0)
dx_per_lesion = df.groupby("lesion_id")["dx"].nunique()
print("Lesions with more than one dx:", int((dx_per_lesion > 1).sum()))

# Do metadata rows and image files match in both directions?
files = {f[:-4] for f in os.listdir(IMAGES_DIR) if f.lower().endswith(".jpg")}
other_files = [f for f in os.listdir(IMAGES_DIR) if not f.lower().endswith(".jpg")]
missing_files = set(df["image_id"]) - files
orphan_files  = files - set(df["image_id"])
print(f"\nJPG files: {len(files)} | metadata rows without a file: {len(missing_files)} | files without a metadata row: {len(orphan_files)}")
print("Non-JPG files in the folder:", other_files[:10])

SUMMARY["missing"] = {c: int(quality.loc[c, "NaN"] + quality.loc[c, "'unknown'"]) for c in quality.index}
SUMMARY["missing_files"] = len(missing_files)
SUMMARY["orphan_files"] = len(orphan_files)
SUMMARY["lesions_with_multiple_dx"] = int((dx_per_lesion > 1).sum())

## 3. Class balance

This shows how uneven the classes are, and sets the **baseline**: a "model" that always answers *nevus* gets this accuracy without looking at a single image. Any real result must be compared against it, per class.

The last column reproduces the class weights used in `TRAIN.ipynb` (`1 / count`), rescaled so they average to 1 for readability.

In [ ]:
counts = df["dx"].value_counts()
bal = pd.DataFrame({
    "name": [CLASS_NAMES[c] for c in counts.index],
    "images": counts.values,
    "% of data": (counts.values / len(df) * 100).round(2),
}, index=counts.index)
w = 1.0 / bal["images"]
bal["loss weight (normalised)"] = (w / w.mean()).round(3)
display(bal)

majority_acc = counts.iloc[0] / len(df)
imbalance = counts.max() / counts.min()
print(f"Majority-class baseline accuracy: {majority_acc:.2%}  (always predict '{counts.index[0]}')")
print(f"Imbalance ratio largest/smallest: {imbalance:.1f}x")
print(f"Malignant share (mel+bcc+akiec): {df['dx'].isin(MALIGNANT).mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
colors = ["#d64545" if c in MALIGNANT else "#e2701f" for c in counts.index]
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title("Images per class (red = malignant)")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v}", ha="center", va="bottom", fontsize=8)
axes[1].bar(counts.index, counts.values, color=colors)
axes[1].set_yscale("log"); axes[1].set_title("Same, log scale")
save("01_class_balance")

SUMMARY["class_counts"] = counts.to_dict()
SUMMARY["majority_baseline_acc"] = round(float(majority_acc), 4)
SUMMARY["imbalance_ratio"] = round(float(imbalance), 1)

## 4. Lesions, and leakage in the old split

HAM10000 often has **several photos of the same lesion**. If one photo is in training and another in validation, the model is partly tested on something it has already seen, which inflates validation accuracy.

The old `TRAIN.ipynb` split by **image** (`train_test_split(..., stratify=dx, random_state=42)`). Below we reproduce that exact split and measure the overlap, then show that a split grouped by `lesion_id` removes it.

In [ ]:
imgs_per_lesion = df.groupby("lesion_id").size()
n_lesions = imgs_per_lesion.size
multi = (imgs_per_lesion > 1)
print(f"Unique lesions: {n_lesions}  (images: {len(df)})")
print(f"Lesions with more than one image: {multi.sum()} ({multi.mean():.1%})")
print(f"Images that belong to a multi-image lesion: {imgs_per_lesion[multi].sum()} ({imgs_per_lesion[multi].sum()/len(df):.1%})")
print("Images per lesion (count of lesions):", imgs_per_lesion.value_counts().sort_index().to_dict())

lesion_dx = df.groupby("lesion_id")["dx"].first()
per_class = pd.DataFrame({
    "lesions": lesion_dx.value_counts(),
    "images": df["dx"].value_counts(),
})
per_class["images per lesion"] = (per_class["images"] / per_class["lesions"]).round(2)
display(per_class.sort_values("images", ascending=False))

fig, ax = plt.subplots(figsize=(6, 3.2))
vc = imgs_per_lesion.value_counts().sort_index()
ax.bar(vc.index.astype(str), vc.values, color="#e2701f")
ax.set_xlabel("images of the same lesion"); ax.set_ylabel("number of lesions")
ax.set_title("How many photos each lesion has")
save("02_images_per_lesion")

SUMMARY["n_lesions"] = int(n_lesions)
SUMMARY["pct_lesions_multi_image"] = round(float(multi.mean()) * 100, 2)

In [ ]:
# Reproduce the OLD split exactly as in TRAIN.ipynb
old_train, old_val = train_test_split(df, test_size=0.2, stratify=df["dx"], random_state=42)
shared = set(old_train["lesion_id"]) & set(old_val["lesion_id"])
leaky_val = old_val["lesion_id"].isin(shared)
print(f"Old split: train {len(old_train)} | val {len(old_val)}")
print(f"Lesions present in BOTH train and val: {len(shared)}")
print(f"Validation images whose lesion was also in training: {leaky_val.sum()} ({leaky_val.mean():.1%} of validation)")

leak_by_class = old_val.assign(leaky=leaky_val).groupby("dx")["leaky"].mean().mul(100).round(1)
print("\n% of validation images leaked, per class:\n", leak_by_class.sort_values(ascending=False).to_string())

# A lesion-grouped, class-stratified split: 5 folds -> one fold is 20%
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(sgkf.split(df, df["dx"], groups=df["lesion_id"]))
new_tr, new_te = df.iloc[tr_idx], df.iloc[te_idx]
overlap_new = set(new_tr["lesion_id"]) & set(new_te["lesion_id"])
print(f"\nGrouped split: train {len(new_tr)} | held-out {len(new_te)} | shared lesions: {len(overlap_new)}")

props = pd.DataFrame({
    "all data %": df["dx"].value_counts(normalize=True).mul(100).round(2),
    "grouped train %": new_tr["dx"].value_counts(normalize=True).mul(100).round(2),
    "grouped held-out %": new_te["dx"].value_counts(normalize=True).mul(100).round(2),
})
display(props)

SUMMARY["old_split_shared_lesions"] = len(shared)
SUMMARY["old_split_pct_val_leaked"] = round(float(leaky_val.mean()) * 100, 2)
SUMMARY["old_split_pct_val_leaked_by_class"] = leak_by_class.to_dict()
SUMMARY["grouped_split_shared_lesions"] = len(overlap_new)

## 5. `dx_type`: a column that must never be an input

`dx_type` records **how** the diagnosis was confirmed. Suspicious lesions are biopsied (`histo`), harmless-looking ones are often just followed up (`follow_up`). That makes `dx_type` a strong hint about the answer that would not exist for a new patient at prediction time. Using it as a feature would be **label leakage**.

In [ ]:
ct = pd.crosstab(df["dx"], df["dx_type"], normalize="index").mul(100).round(1)
display(ct)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ct.loc[counts.index].plot(kind="barh", stacked=True, ax=ax, colormap="Oranges", edgecolor="white", width=0.75)
ax.set_xlabel("% of images in the class"); ax.set_title("How each diagnosis was confirmed")
ax.legend(title="dx_type", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
save("03_dx_type_by_class")

SUMMARY["dx_type_by_class_pct"] = ct.to_dict(orient="index")

## 6. Patient metadata vs class

These are the three fields the app collects **and** the dataset contains, so they are the only candidates for metadata fusion. If a field barely differs between classes, it can't help the model much.

In [ ]:
# Age
print("Age summary:\n", df["age"].describe().round(1).to_string())
age_by_class = df.groupby("dx")["age"].agg(["count", "mean", "median", "std"]).round(1)
age_by_class["missing"] = df.groupby("dx")["age"].apply(lambda s: int(s.isna().sum()))
display(age_by_class.loc[counts.index])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(df["age"].dropna(), bins=range(0, 95, 5), color="#e2701f", edgecolor="white")
axes[0].set_title("Age distribution"); axes[0].set_xlabel("age")
order = counts.index.tolist()
boxplot(axes[1], [df.loc[df["dx"] == c, "age"].dropna() for c in order], order)
axes[1].set_title("Age by class"); axes[1].set_ylabel("age")
save("04_age")

SUMMARY["age_median_by_class"] = age_by_class["median"].to_dict()

In [ ]:
# Sex
sex_ct = pd.crosstab(df["dx"], df["sex"])
sex_pct = pd.crosstab(df["dx"], df["sex"], normalize="index").mul(100).round(1)
display(sex_ct.loc[counts.index]); display(sex_pct.loc[counts.index])

fig, ax = plt.subplots(figsize=(7, 3.4))
sex_pct.loc[counts.index].plot(kind="bar", ax=ax, color=["#f0a46b", "#c2410c", "#9ca3af"][:sex_pct.shape[1]], width=0.75)
ax.set_ylabel("% of class"); ax.set_title("Sex by class"); ax.legend(frameon=False)
plt.xticks(rotation=0)
save("05_sex_by_class")

SUMMARY["sex_pct_by_class"] = sex_pct.to_dict(orient="index")

In [ ]:
# Body site
loc_counts = df["localization"].value_counts()
print("Body-site values used in HAM10000:\n", loc_counts.to_string())

loc_pct = pd.crosstab(df["localization"], df["dx"], normalize="columns").mul(100)
loc_pct = loc_pct.loc[loc_counts.index, counts.index]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(loc_pct.values, cmap="Oranges", aspect="auto")
ax.set_xticks(range(len(loc_pct.columns)), loc_pct.columns)
ax.set_yticks(range(len(loc_pct.index)), loc_pct.index)
for i in range(loc_pct.shape[0]):
    for j in range(loc_pct.shape[1]):
        v = loc_pct.values[i, j]
        if v >= 1:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if v > 25 else "#3a2a1a")
ax.set_title("Where each class appears (% of the class's images)")
plt.colorbar(im, ax=ax, fraction=0.03)
save("06_site_by_class")

# The app's form uses different names. This is a suggested mapping for metadata fusion.
APP_TO_HAM = {
    "Anterior Trunk": ["trunk", "chest", "abdomen"],
    "Posterior Trunk": ["back"],
    "Upper Extremity": ["upper extremity"],
    "Lower Extremity": ["lower extremity"],
    "Face & Neck": ["face", "neck"],
    "Scalp": ["scalp"], "Hand": ["hand"], "Foot": ["foot"],
    "Genital": ["genital"], "Ear": ["ear"], "Acral": ["acral"],
}
mapped = {v for vs in APP_TO_HAM.values() for v in vs}
print("\nHAM sites the app form can't express:", sorted(set(loc_counts.index) - mapped))
print("Mapped names that don't exist in HAM:", sorted(mapped - set(loc_counts.index)))

SUMMARY["sites"] = loc_counts.to_dict()

## 7. Image properties (one pass over every image)

For each image this computes: size and colour mode, file size, mean and spread of each colour channel, brightness, a **dark-corner score** (dermoscopes often leave black circular corners), and a 16×16 **perceptual hash** used to find near-duplicate photos.

Results are cached to `image_stats.csv`, so this only runs once.

In [ ]:
STATS_CSV = f"{OUT_DIR}/image_stats.csv"

def image_stats(image_id):
    path = f"{IMAGES_DIR}/{image_id}.jpg"
    try:
        with Image.open(path) as im:
            w, h, mode = im.width, im.height, im.mode
            small = im.convert("RGB").resize((128, 96))
    except Exception as e:
        return {"image_id": image_id, "error": str(e)}
    a = np.asarray(small, dtype=np.float32) / 255.0
    gray = a.mean(axis=2)
    ch, cw = gray.shape[0] // 8, gray.shape[1] // 8
    corners = np.concatenate([gray[:ch, :cw].ravel(), gray[:ch, -cw:].ravel(),
                              gray[-ch:, :cw].ravel(), gray[-ch:, -cw:].ravel()])
    tiny = np.asarray(small.convert("L").resize((16, 16)), dtype=np.float32)
    bits = (tiny > tiny.mean()).ravel()
    ahash = "".join("1" if b else "0" for b in bits)
    return {
        "image_id": image_id, "width": w, "height": h, "mode": mode,
        "file_kb": round(os.path.getsize(path) / 1024, 1),
        "mean_r": a[..., 0].mean(), "mean_g": a[..., 1].mean(), "mean_b": a[..., 2].mean(),
        "std_r": a[..., 0].std(), "std_g": a[..., 1].std(), "std_b": a[..., 2].std(),
        "brightness": gray.mean(), "corner_brightness": corners.mean(),
        "ahash": f"{int(ahash, 2):064x}", "error": "",
    }

if os.path.exists(STATS_CSV):
    stats = pd.read_csv(STATS_CSV, dtype={"ahash": str})
    print("Loaded cached image stats:", stats.shape)
else:
    ids = df["image_id"].tolist()
    with ThreadPoolExecutor(max_workers=16) as ex:
        rows = list(tqdm(ex.map(image_stats, ids), total=len(ids), desc="Reading images"))
    stats = pd.DataFrame(rows)
    stats.to_csv(STATS_CSV, index=False)
    print("Saved:", STATS_CSV, stats.shape)

stats["error"] = stats["error"].fillna("")
bad = stats[stats["error"] != ""]
print("Unreadable images:", len(bad))
img = df.merge(stats[stats["error"] == ""], on="image_id")
SUMMARY["unreadable_images"] = int(len(bad))

In [ ]:
sizes = (img["width"].astype(int).astype(str) + "×" + img["height"].astype(int).astype(str)).value_counts()
print("Image sizes:\n", sizes.head(10).to_string())
print("\nColour modes:", img["mode"].value_counts().to_dict())
print(f"File size: median {img['file_kb'].median():.0f} KB, range {img['file_kb'].min():.0f}–{img['file_kb'].max():.0f} KB")

# How much of the image does the inference-time center crop keep?
w0, h0 = [int(x) for x in sizes.index[0].split("×")]
short = min(w0, h0); kept = (224 / 256 * short) ** 2 / (w0 * h0)
print(f"\nFor the most common size {w0}×{h0}: Resize(256)+CenterCrop(224) keeps {kept:.0%} of the image area.")
print(f"The old training transform Resize((256,256)) changes the aspect ratio from {w0/h0:.2f} to 1.00.")

SUMMARY["image_sizes_top"] = sizes.head(5).to_dict()
SUMMARY["center_crop_area_kept_pct"] = round(kept * 100, 1)

In [ ]:
# Colour statistics of THIS dataset vs ImageNet (what the pretrained model expects)
ds_mean = img[["mean_r", "mean_g", "mean_b"]].mean().values
ds_std = img[["std_r", "std_g", "std_b"]].mean().values
comp = pd.DataFrame({"HAM10000 mean": ds_mean, "ImageNet mean": [0.485, 0.456, 0.406],
                     "HAM10000 std (avg per image)": ds_std, "ImageNet std": [0.229, 0.224, 0.225]},
                    index=["R", "G", "B"]).round(3)
display(comp)
print("Note: per-image std underestimates the dataset-wide std; it's a rough comparison.")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
boxplot(axes[0], [img.loc[img["dx"] == c, "brightness"] for c in order], order)
axes[0].set_title("Brightness by class"); axes[0].set_ylabel("mean brightness (0–1)")
mean_rgb = img.groupby("dx")[["mean_r", "mean_g", "mean_b"]].mean().loc[order]
for i, c in enumerate(order):
    axes[1].add_patch(plt.Rectangle((i, 0), 0.9, 1, color=tuple(mean_rgb.loc[c].values)))
axes[1].set_xlim(0, len(order)); axes[1].set_ylim(0, 1)
axes[1].set_xticks(np.arange(len(order)) + 0.45, order); axes[1].set_yticks([])
axes[1].set_title("Average colour per class")
save("07_colour_by_class")

SUMMARY["dataset_mean_rgb"] = [round(float(x), 3) for x in ds_mean]

In [ ]:
# Dark dermoscope corners: if one class has them much more often, the model can learn the CORNERS instead of the lesion
DARK = 0.15
img["dark_corners"] = img["corner_brightness"] < DARK
dark_by_class = img.groupby("dx")["dark_corners"].mean().mul(100).round(1).loc[order]
print(f"Images with dark corners (corner brightness < {DARK}): {img['dark_corners'].mean():.1%}")
print("By class (%):\n", dark_by_class.to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(order, dark_by_class.values, color="#e2701f")
ax.set_ylabel("% of images"); ax.set_title("Dark-corner images per class (possible shortcut)")
save("08_dark_corners")

SUMMARY["dark_corner_pct_by_class"] = dark_by_class.to_dict()

In [ ]:
# Near-duplicates: same 16×16 perceptual hash AND nearly the same average colour.
# The interesting case is the SAME photo under DIFFERENT lesion IDs, because a lesion-grouped split can't catch those.
img["colour_key"] = (img[["mean_r", "mean_g", "mean_b"]] * 50).round().astype(int).astype(str).agg("-".join, axis=1)
img["dup_key"] = img["ahash"] + "_" + img["colour_key"]
groups = img.groupby("dup_key").filter(lambda g: len(g) > 1)
hash_groups = groups.groupby("dup_key")
cross_lesion = [g for _, g in hash_groups if g["lesion_id"].nunique() > 1]
cross_label = [g for g in cross_lesion if g["dx"].nunique() > 1]
print(f"Hash groups with 2+ images: {hash_groups.ngroups}")
print(f"...spanning different lesion_ids: {len(cross_lesion)}   ...with different labels: {len(cross_label)}")
print("A matching hash is a strong hint, not proof. Look at the pairs below before concluding anything.")

show = cross_lesion[:4]
if show:
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 2.6 * len(show)), squeeze=False)
    for r, g in enumerate(show):
        for c in range(2):
            row = g.iloc[c]
            axes[r, c].imshow(Image.open(f"{IMAGES_DIR}/{row['image_id']}.jpg"))
            axes[r, c].set_title(f"{row['image_id']} | {row['lesion_id']} | {row['dx']}", fontsize=7)
            axes[r, c].axis("off")
    save("09_possible_duplicates")

SUMMARY["hash_groups_cross_lesion"] = len(cross_lesion)
SUMMARY["hash_groups_cross_label"] = len(cross_label)

## 8. Sample images per class

Look for differences a model could latch onto that aren't the lesion: rulers, ink marks, hair, gel bubbles, dark corners, colour casts.

In [ ]:
N = 6
fig, axes = plt.subplots(len(order), N, figsize=(N * 1.9, len(order) * 1.6))
for r, c in enumerate(order):
    sample = df[df["dx"] == c].sample(N, random_state=SEED)
    for k, (_, row) in enumerate(sample.iterrows()):
        ax = axes[r, k]
        ax.imshow(Image.open(f"{IMAGES_DIR}/{row['image_id']}.jpg"))
        ax.set_xticks([]); ax.set_yticks([])
        if k == 0:
            ax.set_ylabel(c, rotation=0, labelpad=22, fontsize=10, fontweight="bold")
plt.suptitle("Random samples per class", y=1.0)
plt.tight_layout()
save("10_samples_per_class")

## 9. What the model actually sees

Left: the original photo, with the region kept at inference marked. Middle: the **old training** view (`Resize((256,256))` squashes the photo into a square, then a random 224 crop). Right: the **validation / app** view (`Resize(256)` keeps proportions, then the center 224 is kept).

If the middle and right images look noticeably different in shape, the model was trained on one geometry and tested on another.

In [ ]:
def old_train_view(im, rng):
    sq = im.resize((256, 256))
    x, y = rng.integers(0, 33), rng.integers(0, 33)
    return sq.crop((x, y, x + 224, y + 224))

def eval_view(im):
    w, h = im.size; s = 256 / min(w, h)
    rs = im.resize((round(w * s), round(h * s)))
    W, H = rs.size; l, t = int(round((W - 224) / 2)), int(round((H - 224) / 2))
    return rs.crop((l, t, l + 224, t + 224)), (l / s, t / s, (l + 224) / s, (t + 224) / s)

rng = np.random.default_rng(SEED)
ids = df.groupby("dx").sample(1, random_state=SEED)["image_id"].tolist()[:4]
fig, axes = plt.subplots(len(ids), 3, figsize=(10, 3 * len(ids)))
for r, iid in enumerate(ids):
    im = Image.open(f"{IMAGES_DIR}/{iid}.jpg").convert("RGB")
    ev, box = eval_view(im)
    boxed = im.copy(); ImageDraw.Draw(boxed).rectangle(box, outline=(255, 255, 255), width=4)
    for c, (title, pic) in enumerate([("original + kept region", boxed),
                                      ("old training view (squashed)", old_train_view(im, rng)),
                                      ("validation / app view", ev)]):
        axes[r, c].imshow(pic); axes[r, c].set_title(title, fontsize=9); axes[r, c].axis("off")
plt.tight_layout()
save("11_preprocessing_views")

## 10. Findings → decisions for the training notebook

This cell turns the numbers above into concrete decisions and saves everything to `eda_summary.json`.

In [ ]:
decisions = []
decisions.append(f"Baseline to beat: always predicting 'nv' gives {SUMMARY['majority_baseline_acc']:.1%} accuracy. "
                 f"Report per-class recall and macro-F1, not just accuracy.")
decisions.append(f"Imbalance is {SUMMARY['imbalance_ratio']}x. Keep class-weighted loss (or a weighted sampler) and "
                 f"watch recall on df, vasc and akiec.")
if SUMMARY["old_split_shared_lesions"] > 0:
    decisions.append(f"The old split leaked: {SUMMARY['old_split_pct_val_leaked']}% of validation images had a sibling "
                     f"photo of the same lesion in training. Split by lesion_id (StratifiedGroupKFold).")
decisions.append("Never use dx_type as an input: it encodes how the label was obtained.")
if SUMMARY["missing"].get("age", 0) > 0:
    decisions.append(f"Age is missing for {SUMMARY['missing']['age']} images. For fusion: impute the median and add an "
                     f"'age missing' flag.")
decisions.append("Map the app's body-site options to HAM10000's values with one shared mapping used in training "
                 "and inference.")
decisions.append(f"Inference keeps about {SUMMARY['center_crop_area_kept_pct']}% of a typical image. Use the same "
                 f"aspect-preserving geometry in training (RandomResizedCrop on the unsquashed image) so train and "
                 f"test match.")
dc = SUMMARY["dark_corner_pct_by_class"]
if max(dc.values()) - min(dc.values()) > 15:
    decisions.append(f"Dark corners are unevenly spread across classes ({min(dc.values()):.0f}%–{max(dc.values()):.0f}%). "
                     f"Check Grad-CAM for corner attention; consider cropping corners or using them in augmentation.")
if SUMMARY["hash_groups_cross_lesion"] > 0:
    decisions.append(f"{SUMMARY['hash_groups_cross_lesion']} possible duplicate groups span different lesion_ids. "
                     f"Inspect them; if real, merge their groups before splitting.")

print("DECISIONS\n" + "\n".join(f"{i}. {d}" for i, d in enumerate(decisions, 1)))
SUMMARY["decisions"] = decisions
SUMMARY["figures"] = [os.path.basename(f) for f in FIGS]

with open(f"{OUT_DIR}/eda_summary.json", "w") as f:
    json.dump(SUMMARY, f, indent=2, default=str)
print(f"\nSaved {OUT_DIR}/eda_summary.json and {len(FIGS)} figures.")